In [1]:
%load_ext autoreload
%autoreload 2

import gc
import json
import sys
from datetime import date
from pathlib import Path

import mne
import numpy as np
import optuna
import torch
import yaml
from torch.utils.data import DataLoader

root = Path.cwd().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from lib.data import TFRDataset
from lib.models.alexnet import AlexNetTFR
from lib.models.tfr_transformer import TFRTransformerWrapper
from lib.optuna import (
    attrs_fn,
    cumulative_loss_metric_factory,
    loss_slope,
    make_objective_engine,
    make_splits_fn_factory,
    objectives_fn,
    params_fn_factory,
    params_fn_factory_transformer,
    run_fold_fn_factory,
)
from lib.training.epochs import eval_one_epoch_f1_macro, train_one_epoch
from lib.data.normalisation import normalize_tfr_robust

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("project root:", root)
print("device:", device)

project root: /beegfs/home/t.samsonov/notebooks/Pirogov/NeuronDeCo
device: cuda


In [2]:
import warnings
warnings.filterwarnings('ignore') # Suppress all warnings

In [3]:
patient_list = ['s02','s03','s04','s05','s06','s07','s09','s10', 's11','s12','s13','s15']

# Рабочая директория: .../Pirogov/NeuronDeCo/notebooks
NB_DIR = Path.cwd().resolve()
PROJECT_ROOT = NB_DIR.parent
PIROGOV_ROOT = PROJECT_ROOT.parent
PREPROCESSED_ROOT = PIROGOV_ROOT / "PreprocessedData"

patients_candidates = [
    PREPROCESSED_ROOT / "patients.yaml",  # приоритет: как вы просили
    PROJECT_ROOT / "patients.yaml",       # fallback на старый вариант
]
patients_yaml = next((p for p in patients_candidates if p.exists()), None)
if patients_yaml is None:
    raise FileNotFoundError(
        "patients.yaml not found. Checked: " + ", ".join(str(p) for p in patients_candidates)
    )

with open(patients_yaml, encoding="utf-8") as f:
    test = yaml.safe_load(f)
pat_config = test["default"]

min_f, max_f, min_t, max_t = (
    pat_config["min_f"],
    pat_config["max_f"],
    pat_config["min_t"],
    pat_config["max_t"],
)

cumulative_weighted_loss_metric = cumulative_loss_metric_factory(
    up_weight=1.1,
    down_weight=1.0,
)

In [5]:
seed = 42
cv = True
test_size = 0.2
cv_aggregate = "median"
max_epochs = 100
patience = 10
n_trials = 10

In [6]:
for s in patient_list:
    # time_frequency_file = Path(pat_config["time_frequency_file"])
    tfr_candidates = [
        PREPROCESSED_ROOT / "specs_with_car" / f"tfr_{s}.fif",  # приоритет: PreprocessedData
        PROJECT_ROOT / "specs_with_car" / f"tfr_{s}.fif",       # fallback
    ]
    time_frequency_file = next((p for p in tfr_candidates if p.exists()), None)
    if time_frequency_file is None:
        raise FileNotFoundError(
            "TFR file not found. Checked: " + ", ".join(str(p) for p in tfr_candidates)
        )
    DATA_ROOT_ROOT = time_frequency_file.parent.parent #  Хардкод структуры
    DATE_TAG = date.today().isoformat()
    OPTUNA_DB_DIR = DATA_ROOT_ROOT / DATE_TAG
    OPTUNA_DB_DIR.mkdir(parents=True, exist_ok=True)

    print("notebooks dir:", NB_DIR)
    print("Pirogov root:", PIROGOV_ROOT)
    print("PreprocessedData root:", PREPROCESSED_ROOT)
    print("patients.yaml:", patients_yaml)
    print("TFR file:", time_frequency_file)
    print("Optuna DB dir:", OPTUNA_DB_DIR)
    print("config slices:", (min_f, max_f, min_t, max_t))

    tfr = mne.time_frequency.read_tfrs(str(time_frequency_file))[0]
    y = np.where(tfr.events[:, 2] == 9, 1, 0).astype(np.int64)

    # X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, min_f:max_f, min_t:max_t]
    X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, :-50, :].astype(np.float32)

    del tfr
    gc.collect()  # Теперь память гарантировано свободна

    n, c, f, t = X.shape
    num_classes = int(np.unique(y).shape[0])
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("num_classes:", num_classes)

    # ==================================
    # Alexnet
    # ==================================

    alex_db = OPTUNA_DB_DIR / f"{time_frequency_file.stem}_alexnet.db"
    alex_storage = f"sqlite:///{alex_db}"

    objective_alex = make_objective_engine(
        X=X,
        y=y,
        make_splits_fn=make_splits_fn_factory(test_size=test_size, seed=seed, cv=cv),
        run_fold_fn=run_fold_fn_factory(
            ModelCls=AlexNetTFR,
            device=device,
            max_epochs=max_epochs,
            patience=patience,
            TFRDataset=TFRDataset,
            DataLoader=DataLoader,
            train_one_epoch=train_one_epoch,
            eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
            loss_metric=cumulative_weighted_loss_metric,
        ),
        aggregate_mode=cv_aggregate,
        params_fn=params_fn_factory(in_channels=c, num_classes=num_classes),
        objectives_fn=objectives_fn,
        attrs_fn=attrs_fn,
    )

    study_a = optuna.create_study(
        directions=["maximize", "minimize"],
        sampler=optuna.samplers.NSGAIISampler(seed=seed),
        storage=alex_storage,
        study_name=f"{time_frequency_file.stem}_alexnet",
        load_if_exists=True,
    )
    study_a.optimize(objective_alex, n_trials=n_trials, show_progress_bar=True)

    print("AlexNet DB:", alex_db)


notebooks dir: /beegfs/home/t.samsonov/notebooks/Pirogov/NeuronDeCo/notebooks
Pirogov root: /beegfs/home/t.samsonov/notebooks/Pirogov
PreprocessedData root: /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData
patients.yaml: /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/patients.yaml
TFR file: /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/specs_with_car/tfr_s02.fif
Optuna DB dir: /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/2026-03-31
config slices: (5, 30, 100, -400)
Reading /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/specs_with_car/tfr_s02.fif ...
Not setting metadata
X shape: (236, 3, 50, 1001)
y shape: (236,)
num_classes: 2


[I 2026-03-31 11:46:50,109] Using an existing study with name 'tfr_s02_alexnet' instead of creating a new one.


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-03-31 11:47:29,054] Trial 2 finished with values: [0.7655328798185941, 0.5402824989024629] and parameters: {'batch_size': 32, 'dropout': 0.4190609389379256, 'lr': 2.4348773534554605e-05, 'weight_decay': 4.207053950287936e-06}.
[I 2026-03-31 11:48:05,499] Trial 3 finished with values: [0.7445652173913043, 0.5865787977867939] and parameters: {'batch_size': 32, 'dropout': 0.4956508044572318, 'lr': 1.1245798259119336e-05, 'weight_decay': 0.00757947995334801}.
[I 2026-03-31 11:48:39,083] Trial 4 finished with values: [0.7655328798185941, 0.958080182430592] and parameters: {'batch_size': 16, 'dropout': 0.12838315689740368, 'lr': 5.670807781371427e-05, 'weight_decay': 0.0001256104370001356}.
[I 2026-03-31 11:49:13,352] Trial 5 finished with values: [0.7445652173913043, 0.5833581328392029] and parameters: {'batch_size': 64, 'dropout': 0.09764570245642928, 'lr': 5.292705365436973e-05, 'weight_decay': 2.9204338471814107e-05}.
[I 2026-03-31 11:49:47,090] Trial 6 finished with values: [0.7

[I 2026-03-31 11:53:28,146] A new study created in RDB with name: tfr_s03_alexnet


...

  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-03-31 12:54:14,009] Trial 0 finished with values: [0.7991071428571428, 0.6578345537185668] and parameters: {'batch_size': 32, 'dropout': 0.4190609389379256, 'lr': 2.4348773534554605e-05, 'weight_decay': 4.207053950287936e-06}.
[I 2026-03-31 12:54:43,566] Trial 1 finished with values: [0.699666295884316, 0.6077827215194702] and parameters: {'batch_size': 32, 'dropout': 0.4956508044572318, 'lr': 1.1245798259119336e-05, 'weight_decay': 0.00757947995334801}.
[I 2026-03-31 12:55:05,521] Trial 2 finished with values: [0.7963800904977376, 0.9602461930116016] and parameters: {'batch_size': 16, 'dropout': 0.12838315689740368, 'lr': 5.670807781371427e-05, 'weight_decay': 0.0001256104370001356}.
[I 2026-03-31 12:55:31,196] Trial 3 finished with values: [0.7664071190211346, 0.6706850051879882] and parameters: {'batch_size': 64, 'dropout': 0.09764570245642928, 'lr': 5.292705365436973e-05, 'weight_decay': 2.9204338471814107e-05}.
[I 2026-03-31 12:56:01,417] Trial 4 finished with values: [0.7

[I 2026-03-31 12:58:22,013] A new study created in RDB with name: tfr_s15_alexnet


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-03-31 12:58:53,945] Trial 0 finished with values: [0.7105393369619001, 0.638661790556378] and parameters: {'batch_size': 32, 'dropout': 0.4190609389379256, 'lr': 2.4348773534554605e-05, 'weight_decay': 4.207053950287936e-06}.
[I 2026-03-31 12:59:38,539] Trial 1 finished with values: [0.775, 0.6222881020439995] and parameters: {'batch_size': 32, 'dropout': 0.4956508044572318, 'lr': 1.1245798259119336e-05, 'weight_decay': 0.00757947995334801}.
[I 2026-03-31 13:00:09,068] Trial 2 finished with values: [0.73, 0.9990363958089249] and parameters: {'batch_size': 16, 'dropout': 0.12838315689740368, 'lr': 5.670807781371427e-05, 'weight_decay': 0.0001256104370001356}.
[I 2026-03-31 13:00:45,066] Trial 3 finished with values: [0.7163865546218487, 0.872989523410797] and parameters: {'batch_size': 64, 'dropout': 0.09764570245642928, 'lr': 5.292705365436973e-05, 'weight_decay': 2.9204338471814107e-05}.
[I 2026-03-31 13:01:13,676] Trial 4 finished with values: [0.6933333333333334, 1.091331272

In [7]:
seed = 42
cv = True
test_size = 0.2
cv_aggregate = "median"
max_epochs = 100
patience = 10
n_trials = 15

In [6]:
patient_list = ['s12','s13','s15']

In [8]:
for s in patient_list:
    # time_frequency_file = Path(pat_config["time_frequency_file"])
    tfr_candidates = [
        PREPROCESSED_ROOT / "specs_with_car" / f"tfr_{s}.fif",  # приоритет: PreprocessedData
        PROJECT_ROOT / "specs_with_car" / f"tfr_{s}.fif",       # fallback
    ]
    time_frequency_file = next((p for p in tfr_candidates if p.exists()), None)
    if time_frequency_file is None:
        raise FileNotFoundError(
            "TFR file not found. Checked: " + ", ".join(str(p) for p in tfr_candidates)
        )
    DATA_ROOT_ROOT = time_frequency_file.parent.parent #  Хардкод структуры
    DATE_TAG = date.today().isoformat()
    OPTUNA_DB_DIR = DATA_ROOT_ROOT / DATE_TAG
    OPTUNA_DB_DIR.mkdir(parents=True, exist_ok=True)

    print("notebooks dir:", NB_DIR)
    print("Pirogov root:", PIROGOV_ROOT)
    print("PreprocessedData root:", PREPROCESSED_ROOT)
    print("patients.yaml:", patients_yaml)
    print("TFR file:", time_frequency_file)
    print("Optuna DB dir:", OPTUNA_DB_DIR)
    print("config slices:", (min_f, max_f, min_t, max_t))

    tfr = mne.time_frequency.read_tfrs(str(time_frequency_file))[0]
    y = np.where(tfr.events[:, 2] == 9, 1, 0).astype(np.int64)

    # X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, min_f:max_f, min_t:max_t]
    X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, :-50, :].astype(np.float32)

    del tfr
    gc.collect()  # Теперь память гарантировано свободна

    n, c, f, t = X.shape
    num_classes = int(np.unique(y).shape[0])
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("num_classes:", num_classes)

    # ==================================
    # Transformer
    # ==================================
    tr_db = OPTUNA_DB_DIR / f"{time_frequency_file.stem}_transformer.db"
    tr_storage = f"sqlite:///{tr_db}"

    objective_tr = make_objective_engine(
        X=X,
        y=y,
        make_splits_fn=make_splits_fn_factory(test_size=test_size, seed=seed, cv=cv),
        run_fold_fn=run_fold_fn_factory(
            ModelCls=TFRTransformerWrapper,
            device=device,
            max_epochs=max_epochs,
            patience=patience,
            TFRDataset=TFRDataset,
            DataLoader=DataLoader,
            train_one_epoch=train_one_epoch,
            eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
            loss_metric=cumulative_weighted_loss_metric,
        ),
        aggregate_mode=cv_aggregate,
        params_fn=params_fn_factory_transformer(
            num_classes=num_classes,
            seq_len=t,
        ),
        objectives_fn=objectives_fn,
        attrs_fn=attrs_fn,
    )

    study_t = optuna.create_study(
        directions=["maximize", "minimize"],
        sampler=optuna.samplers.NSGAIISampler(seed=seed + 1),
        storage=tr_storage,
        study_name=f"{time_frequency_file.stem}_transformer",
        load_if_exists=True,
    )
    study_t.optimize(objective_tr, n_trials=n_trials, show_progress_bar=True)

    print("Transformer DB:", tr_db)

notebooks dir: /beegfs/home/t.samsonov/notebooks/Pirogov/NeuronDeCo/notebooks
Pirogov root: /beegfs/home/t.samsonov/notebooks/Pirogov
PreprocessedData root: /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData
patients.yaml: /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/patients.yaml
TFR file: /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/specs_with_car/tfr_s12.fif
Optuna DB dir: /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/2026-03-31
config slices: (5, 30, 100, -400)
Reading /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/specs_with_car/tfr_s12.fif ...
Not setting metadata
X shape: (134, 2, 50, 1001)
y shape: (134,)
num_classes: 2


[I 2026-03-31 14:16:14,126] A new study created in RDB with name: tfr_s12_transformer


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-03-31 14:17:34,708] Trial 0 finished with values: [0.5903448275862069, 0.7059876501560212] and parameters: {'embed_dim': 64, 'preprocess': 'channel_conv', 'batch_size': 32, 'dropout': 0.5010235593143333, 'nhead': 8, 'dim_fc': 512, 'num_layers': 7, 'encoder_dropout': 0.5843058002894633, 'mlp_dropout': 0.26195730291732594, 'use_conv': False, 'conv_kernel_size': 7, 'conv_dropout': 0.17880210041325761, 'pooling': 'mean', 'lr': 0.00022160445973744421, 'weight_decay': 7.987032801740347e-05}.
[I 2026-03-31 14:18:06,376] Trial 1 finished with values: [0.554945054945055, 0.6968741059303283] and parameters: {'embed_dim': 64, 'preprocess': 'pixel_weight', 'batch_size': 32, 'dropout': 0.4143928979410806, 'nhead': 4, 'dim_fc': 64, 'num_layers': 4, 'encoder_dropout': 0.4498433955823936, 'mlp_dropout': 0.26955828724724107, 'use_conv': True, 'conv_kernel_size': 3, 'conv_dropout': 0.11949688597362113, 'pooling': 'mean', 'lr': 0.0004215988942013613, 'weight_decay': 1.579308974316983e-05}.
[I 202

[I 2026-03-31 14:27:49,336] A new study created in RDB with name: tfr_s13_transformer


...

[I 2026-03-31 14:29:05,449] Trial 0 finished with values: [0.7321428571428572, 0.716736203432083] and parameters: {'embed_dim': 64, 'preprocess': 'channel_conv', 'batch_size': 32, 'dropout': 0.5010235593143333, 'nhead': 8, 'dim_fc': 512, 'num_layers': 7, 'encoder_dropout': 0.5843058002894633, 'mlp_dropout': 0.26195730291732594, 'use_conv': False, 'conv_kernel_size': 7, 'conv_dropout': 0.17880210041325761, 'pooling': 'mean', 'lr': 0.00022160445973744421, 'weight_decay': 7.987032801740347e-05}.
[I 2026-03-31 14:29:50,204] Trial 1 finished with values: [0.7321428571428572, 0.7025002300739288] and parameters: {'embed_dim': 64, 'preprocess': 'pixel_weight', 'batch_size': 32, 'dropout': 0.4143928979410806, 'nhead': 4, 'dim_fc': 64, 'num_layers': 4, 'encoder_dropout': 0.4498433955823936, 'mlp_dropout': 0.26955828724724107, 'use_conv': True, 'conv_kernel_size': 3, 'conv_dropout': 0.11949688597362113, 'pooling': 'mean', 'lr': 0.0004215988942013613, 'weight_decay': 1.579308974316983e-05}.
[I 202

[I 2026-03-31 14:42:07,349] A new study created in RDB with name: tfr_s15_transformer


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-03-31 14:44:14,758] Trial 0 finished with values: [0.6296296296296297, 0.7331655000315774] and parameters: {'embed_dim': 64, 'preprocess': 'channel_conv', 'batch_size': 32, 'dropout': 0.5010235593143333, 'nhead': 8, 'dim_fc': 512, 'num_layers': 7, 'encoder_dropout': 0.5843058002894633, 'mlp_dropout': 0.26195730291732594, 'use_conv': False, 'conv_kernel_size': 7, 'conv_dropout': 0.17880210041325761, 'pooling': 'mean', 'lr': 0.00022160445973744421, 'weight_decay': 7.987032801740347e-05}.
[I 2026-03-31 14:45:07,240] Trial 1 finished with values: [0.6737588652482269, 0.6839357641008165] and parameters: {'embed_dim': 64, 'preprocess': 'pixel_weight', 'batch_size': 32, 'dropout': 0.4143928979410806, 'nhead': 4, 'dim_fc': 64, 'num_layers': 4, 'encoder_dropout': 0.4498433955823936, 'mlp_dropout': 0.26955828724724107, 'use_conv': True, 'conv_kernel_size': 3, 'conv_dropout': 0.11949688597362113, 'pooling': 'mean', 'lr': 0.0004215988942013613, 'weight_decay': 1.579308974316983e-05}.
[I 20

KeyboardInterrupt: 